In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Data set to be used for the homework
hwlink = "https://www.fuw.edu.pl/~zarnecki/SAED/OECD_Poland_filtered.csv"

# Reading from the web (relevant columns only)
hw= pd.read_csv(hwlink, encoding='latin-1', sep=",")
                
                #usecols=['SEX','Sex','AGE','Age','TIME_PERIOD','OBS_VALUE'])  # Columns are counted from 0 (!)

print(hw)

     SEX     Sex     AGE                  Age  TIME_PERIOD  OBS_VALUE
0      F  Female  Y_GE50     50 years or over         1960  3379322.0
1      F  Female  Y_GE50     50 years or over         1961  3403900.0
2      F  Female  Y_GE50     50 years or over         1962  3502400.0
3      F  Female  Y_GE50     50 years or over         1963  3625100.0
4      F  Female  Y_GE50     50 years or over         1964  3728901.0
...   ..     ...     ...                  ...          ...        ...
3643   F  Female  Y_GE50     50 years or over         2025  8215716.0
3644   F  Female   Y_LE4      4 years or less         2025   707100.0
3645   F  Female  Y55T59  From 55 to 59 years         2025  1115886.0
3646   F  Female  Y30T34  From 30 to 34 years         2025  1198754.0
3647   F  Female  Y15T19  From 15 to 19 years         2025   987098.0

[3648 rows x 6 columns]


## Age categories in the data set

For every year (1950-2025) and sex, the data set gives the population in 24 age categories (`AGE` column):

- **5-year groups**: `Y_LE4` (0-4), `Y5T9` (5-9), ..., `Y80T84` (80-84), plus the open-ended group `Y_GE85` (85 and over). These 18 groups do not overlap, and their sum is the total population.
- **Aggregates**: `Y_LT20` (< 20), `Y15T64`, `Y20T64`, `Y_GE50` (50+), `Y_GE65` (65+) and `_T` (total). These overlap with the 5-year groups and carry no extra information, so we do not use them.

Ages are known only with a resolution of 5 years. A cohort of people born within a 5-year window can therefore be followed in the data only every 5 years. The people born in 1946-1950 are in the group 0-4 in 1950, 5-9 in 1955, ..., 75-79 in 2025: in year $1950 + 5k$ they are the group $[5k,\, 5k+4]$. In the intermediate years the cohort is split between two neighbouring groups.

The histograms below show the age structure in four years, with the bar of the 1946-1950 cohort highlighted.

In [ ]:
# The 5-year groups 0-4, 5-9, ..., 80-84 plus the open group 85+ form a partition
# of the population; all the other categories are overlapping aggregates of them.
groups = ["Y_LE4"] + [f"Y{a}T{a+4}" for a in range(5, 85, 5)] + ["Y_GE85"]
labels = ["0-4"] + [f"{a}-{a+4}" for a in range(5, 85, 5)] + ["85+"]
aggregates = ["Y_LT20", "Y15T64", "Y20T64", "Y_GE50", "Y_GE65", "_T"]

# Age categories available in the data set: 5-year groups in age order, then aggregates
# (sorting by the code itself is alphabetical and would put "Y5T9" after "Y55T59")
cats = (hw[["AGE", "Age"]].drop_duplicates().set_index("AGE")
        .loc[groups + aggregates].reset_index())
cats["type"] = ["5-year group"] * len(groups) + ["aggregate"] * len(aggregates)
print(cats.to_string())

# Check: the sum of the partition reproduces the total population
for sex in ["M", "F"]:
    s = hw[(hw.SEX == sex) & (hw.TIME_PERIOD == 2025)].set_index("AGE").OBS_VALUE
    print(f"{sex} 2025: sum of groups = {s[groups].sum():.0f}, total = {s['_T']:.0f}")

# Age distribution (men + women) in four years: the 1946-1950 cohort (highlighted)
# moves to the next 5-year group every 5 years
years = [1950, 1975, 2000, 2025]
fig, axes = plt.subplots(1, len(years), figsize=(14, 4.5), sharey=True)
for ax, year in zip(axes, years):
    pop = (hw[(hw.TIME_PERIOD == year) & hw.AGE.isin(groups)]
           .groupby("AGE").OBS_VALUE.sum()[groups] / 1e6)
    k = (year - 1950) // 5
    colors = ["#c8c8c8"] * len(groups)
    colors[k] = "#d62728"
    ax.barh(labels, pop, color=colors, edgecolor="white", linewidth=1)
    ax.set_title(f"{year}: cohort aged {labels[k]}")
    ax.set_xlabel("Population [millions]")
    ax.grid(axis="x", alpha=0.3)
axes[0].set_ylabel("Age group [years]")
fig.suptitle("Age structure of Poland (men + women), cohort born 1946-1950 in red")
fig.tight_layout()
plt.show()

In [ ]:
# Task 1: population of men and women born in 1946-1950 as a function of year (and age)

# The 1946-1950 cohort is aged 0-4 in 1950, 5-9 in 1955, ..., 75-79 in 2025:
# in year 1950 + 5k it is in the age group [5k, 5k+4]
rows = []
for k in range(16):
    year = 1950 + 5 * k
    age_code = "Y_LE4" if k == 0 else f"Y{5*k}T{5*k+4}"
    for sex in ["M", "F"]:
        sel = hw[(hw.SEX == sex) & (hw.AGE == age_code) & (hw.TIME_PERIOD == year)]
        rows.append({"year": year, "age_group": f"{5*k}-{5*k+4}", "sex": sex,
                     "population": sel.OBS_VALUE.iloc[0]})

cohort = pd.DataFrame(rows).pivot(index=["year", "age_group"], columns="sex",
                                  values="population").reset_index()
print(cohort)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(cohort.year, cohort.M / 1e6, "o-", label="Men")
ax.plot(cohort.year, cohort.F / 1e6, "s-", label="Women")
ax.set_xlabel("Year")
ax.set_ylabel("Population [millions]")
ax.set_title("Population of Poland born in 1946-1950", pad=35)
ax.grid(alpha=0.3)
ax.legend()

# secondary x axis with the age of the cohort (centre of the age group)
sec = ax.secondary_xaxis("top", functions=(lambda y: y - 1950 + 2, lambda a: a + 1950 - 2))
sec.set_xlabel("Age [years]")

plt.show()